# Types

> Core internal types.

In [ ]:
#| default_exp types

## Imports

In [ ]:
#| export
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

In [ ]:
#| hide
from fastcore.test import *

## Purpose And Design

`types.py` is the canonical data model for fastllm.

### Why this module exists

OpenAI, Anthropic, Gemini, and openai-compatible providers emit different payload structures. This module defines a small, stable internal vocabulary so the rest of the codebase can remain provider-agnostic.

### Architectural fit

- Upstream: provider-specific serializers/parsers (`clients`, `normalize`).
- Downstream: stream collation (`streaming`), cost logic (`costs`), high-level API (`highlevel`), and user-visible results.

### Key design choices

- Immutable dataclasses (`frozen=True`) keep response objects predictable.
- `Part` + `Msg` support canonical multimodal content without committing to provider wire formats.
- `Completion` and `Delta` separate non-stream and stream semantics while sharing core fields (`tool_calls`, `usage`).

### Reader outcome

After this notebook, you should have the core mental model of what every other module produces or consumes.

### `Part`

Canonical atomic content unit for multimodal inputs/outputs.

**Why it exists**
- Providers represent content blocks differently (`input_text`, `text`, `inlineData`, `source`, etc).
- `Part` gives one stable internal shape so serialization/parsing logic can stay at provider boundaries.

**Design Notes**
- `type` encodes semantic modality (`text`, `input_image`, `input_file`, etc).
- `text` is for simple text payloads.
- `data` carries structured provider-agnostic payload details when text is insufficient.

**Connections**
- Wrapped by `Msg.content`.
- Produced/consumed by `highlevel` coercion, provider serializers in `clients`, and normalizers in `normalize`.
- Key enabler for model-only swapping without rewriting message construction code.

In [ ]:
#| export
class Part:
    "A normalized content part."
    type: str
    text: Optional[str] = None
    data: Optional[Dict[str, Any]] = None

### `Msg`

Canonical conversation turn abstraction.

**Why it exists**
- Conversation structures vary across APIs (chat messages, response input items, content blocks).
- `Msg` lets the rest of fastllm reason in one conversation format.

**Design Notes**
- `role` captures turn semantics (`user`, `assistant`, `tool`, etc).
- `content` is a list of `Part` to support multimodal turns consistently.
- `data` stores auxiliary metadata (tool call maps, provider-specific hints) without polluting canonical fields.

**Connections**
- Primary input to `acompletion` and provider clients.
- Used by toolloop replay flows (`StreamSummary`/`Completion` -> `Msg` coercion in `highlevel`).

In [ ]:
#| export
class Msg:
    "A normalized message."
    role: str
    content: List[Part]
    data: Optional[Dict[str, Any]] = None

### `ToolCall`

Canonical tool invocation descriptor.

**Why it exists**
- Providers stream tool calls differently (chunked args, id/index aliases, function blocks).
- `ToolCall` normalizes identity + name + arguments for downstream tool execution.

**Design Notes**
- `id` is the stable join key for tool result messages.
- `arguments` may be fully parsed or reconstructed from deltas during stream collation.

**Connections**
- Emitted by `normalize` and finalized by `streaming.acollect_stream`.
- Consumed by toolloop code in user layers (e.g. lisette-style loops).

In [ ]:
#| export
class ToolCall:
    "Normalized tool call."
    id: str
    name: str
    arguments: Dict[str, Any] = field(default_factory=dict)

### `Usage`

Canonical token/accounting container.

**Why it exists**
- Token usage fields differ by provider (`prompt_tokens`, `input_tokens`, `promptTokenCount`, etc).
- Cost/monitoring code needs one stable representation.

**Design Notes**
- Normalized token totals are first-class fields.
- `raw` preserves full provider payload for advanced accounting fields (cached/reasoning/search token details).

**Connections**
- Produced by usage normalizers in `normalize`.
- Consumed by `costs.estimate_cost` and stream summaries.

In [ ]:
#| export
class Usage:
    "Normalized usage."
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    raw: Dict[str, Any] = field(default_factory=dict)

### `Caps`

Capability declaration for a client.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Core fields: `tools: bool`, `tool_choice: bool`, `streaming: bool`, `search: bool`, `reasoning: bool`, `prefill: bool` ...
- Encapsulates one coherent concept to reduce repeated shape logic elsewhere.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: `clients`.
- Module downstream: `normalize`, `streaming`, `costs`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
class Caps:
    "Capability declaration for a client."
    tools: bool = False
    tool_choice: bool = False
    streaming: bool = True
    search: bool = False
    reasoning: bool = False
    prefill: bool = False
    citations: bool = False
    prompt_caching: bool = False
    images: bool = True
    pdfs: bool = False
    url_context: bool = False

### `RequestOptions`

Unified high-level request option envelope.

**Why it exists**
- Each provider exposes overlapping but non-identical generation options.
- This type defines a canonical option surface with controlled escape hatches.

**Design Notes**
- Canonical knobs: `max_tokens`, `temperature`, `tools`, `tool_choice`, `reasoning_effort`, `cache`.
- Escape hatches: `native`, `extra_body`, `extra_headers`, `extra_query` for full provider coverage.

**Connections**
- Accepted by provider client methods and delegated through `acompletion`.
- Merged/translated in `clients` before request serialization.

In [ ]:
#| export
class RequestOptions:
    "Request options shared across providers."
    max_tokens: Optional[int] = None
    temperature: Optional[float] = None
    cache: Optional[Any] = None
    tools: Optional[List[Dict[str, Any]]] = None
    tool_choice: Optional[Any] = None
    reasoning_effort: Optional[str] = None
    response_format: Optional[Dict[str, Any]] = None
    native: Optional[Dict[str, Any]] = None
    extra_body: Optional[Dict[str, Any]] = None
    extra_headers: Optional[Dict[str, str]] = None
    extra_query: Optional[Dict[str, Any]] = None

### `Completion`

Canonical non-stream model response object.

**Why it exists**
- A uniform response object is required for cross-provider downstream logic.

**Design Notes**
- Contains canonical assistant `message`, optional `tool_calls`, `usage`, and provider `raw` payload.
- Keeps `provider` and `model` explicit for diagnostics and reporting.

**Connections**
- Returned by non-stream client paths and `acompletion(..., stream=False)`.
- Can be coerced back into `Msg` for toolloop continuation (`highlevel`).

In [ ]:
#| export
class Completion:
    "Normalized completion response."
    model: str
    message: Msg
    finish_reason: Optional[str] = None
    usage: Optional[Usage] = None
    tool_calls: List[ToolCall] = field(default_factory=list)
    provider: Optional[str] = None
    raw: Dict[str, Any] = field(default_factory=dict)

### `Delta`

Canonical streaming event fragment.

**Why it exists**
- Stream events arrive incrementally with partial text/args/usage metadata.

**Design Notes**
- Holds incremental text and tool-call fragments.
- Carries `raw` event payload to avoid information loss.

**Connections**
- Produced by provider event normalizers.
- Aggregated by `streaming.acollect_stream` into `StreamSummary`.

In [ ]:
#| export
class Delta:
    "Normalized streaming delta event."
    text: str = ""
    tool_calls: List[ToolCall] = field(default_factory=list)
    finish_reason: Optional[str] = None
    usage: Optional[Usage] = None
    raw: Dict[str, Any] = field(default_factory=dict)

## Quick Checks

In [ ]:
import fastllm.types as m
for nm in ['Part', 'Msg', 'ToolCall', 'Usage', 'Caps', 'RequestOptions', 'Completion', 'Delta']: assert hasattr(m, nm), nm
from fastllm.types import Msg, Part
m = Msg(role='user', content=[Part(type='text', text='hi')])
assert m.content[0].text=='hi'